In [6]:
import copy
import tiktoken

#from settings import Settings
#from models import Question
from src.services.OpenAIClient import (
    OpenAIClient,
    Gpt4o,
    Gpt4oMini,
    #Gpto1Mini,
    #Gpto1Preview,
    Gpto1,
    Gpto3,
    Gpt5,
    Gpt5WithThinking,
)

from src.services.accuracy import calculate_accuracy
from src.services.read_excel import read_excel
##from src.services.recorder import ApiHistoryRecorder
from src.strategies.solve_questions_by_openai import (
    BasicSolveQuestionPrompt,
    solve_questions_by_openai_independently,
    solve_type_d_questions_by_openai_dependently,
    OptimizedSolveQuestionPrompt1,
    NormalSolveQuestionPrompt,
    OptimizedSolveQuestionPrompt2_for_essential_a_b,
    OptimizedSolveQuestionPrompt2_for_c_d
)
from src.strategies.translate_to_English_by_openai import (
    BasicTranslateToEnglishPrompt,
    OptimizedTranslateToEnglishPrompt1,
    NormalTranslateToEnglishPrompt,
    OptimizedTranslateToEnglishPrompt,
)

In [7]:
record_excel_path = "../record/record.xlsx"
file_path = "../problems/75th/essential"
#file_path = "../problems/75th/academic_d"

problem_file_path = f'{file_path}/questions.xlsx'
answer_file_path = f'{file_path}/answers.xlsx'

### =====変更=====
model = Gpt5WithThinking

is_solving_prompt_normal = True
is_translating_prompt_normal = True

EXECUTE_NUM = 1

is_translated_to_English = False

solve_question_prompt = NormalSolveQuestionPrompt if is_solving_prompt_normal else OptimizedSolveQuestionPrompt2_for_c_d # OptimizedSolveQuestionPrompt1
translate_to_english_prompt = NormalTranslateToEnglishPrompt if is_translating_prompt_normal else OptimizedTranslateToEnglishPrompt

is_image_contained = False
### ============

WAIT_SEC = 15 # when the limitation of API request is low, you need to wait for a while between executions.
solve_num = -1 # when this is set to -1, solve all problems in a section
batch_size = 10


is_dry_run = False

is_d = file_path[-1] == 'd'

# === assertion ===
assert is_image_contained == (file_path[-1] in {'c', 'd'})
assert EXECUTE_NUM <= 3



In [8]:
#api_history_recorder = ApiHistoryRecorder(excel_path=record_excel_path)
openai_client = OpenAIClient(
    model=model,
    #api_history_recorder=api_history_recorder,
)

In [9]:
questions = read_excel(file_path=file_path)

for i in range(4):
    print(f'======question:{i+1}======')
    print(questions[i].type_d_common_sentence)
    print(questions[i].question_sentence)
    print(questions[i].answer_options)
    print(questions[i].correct_answer)
    print(questions[i].image_path)


======question:1======

隣家で多数の猫を屋外飼養しており、毎年子猫が生まれて困っているとの相談があった。行政機関の動物愛護業務に携わる獣医師が行う飼い主への対応として適当でないのはどれか。
１．近隣からの苦情を伝え、あとは自己責任で対応するよう伝える。 ２．感染症対策や猫の病気の治療を行っているかを確認する。 ３．不妊去勢手術の必要性について説明し、適正な頭数で飼養するよう伝える。 ４．飼養数を減らすため譲渡先を探すよう伝える。 ５．猫を屋内飼養するよう伝える。
{<AnswerEnum.CHOICE_1: 1>}
None
======question:2======

インフォームドコンセントに関する記述として最も適切なのはどれか。
１．診断や治療の内容（効果やリスクなど）について医療者側が専門用語を駆使 してよく説明することをいう。 ２．獣医療トラブルを防止するために行うものであり、飼い主との関係構築のた めではない。 ３．獣医療行為の効果とリスク、費用に関して十分説明し、理解と合意を得るこ とである。 ４．伴侶動物においては救命を最優先し、飼い主が苦痛の軽減のための安楽死を 望んだとしても獣医師は決して動物の命を奪ってはいけない。 ５．産業動物は経済動物であるため、治療に際し農家の理解を求める必要はない。
{<AnswerEnum.CHOICE_3: 3>}
None
======question:3======

我が国における動物福祉（アニマルウェルフェア）の考え方に即した家畜の輸送に関する記述として正しいのはどれか。
１．短時間の輸送であれば特に配慮は必要ない。
２．長時間の輸送でも乗り物酔い防止のため給餌しない。
３．区画が十分広ければ、異なる動物種を区分して輸送する必要はない。
４．輸送中の責任者として獣医師を任命する必要がある。
５．積込みや積下ろしの際の環境にも配慮する必要がある。
{<AnswerEnum.CHOICE_5: 5>}
None
======question:4======

動物実験における3R原則の組合せとして正しいのはどれか。
１．Reduction（削減） Responsibility（責任） Restriction （制限） ２．Reduction（削減） Responsibility（責任） 

In [10]:
import asyncio

for num in range(EXECUTE_NUM):
    question_temp = copy.deepcopy(questions[:solve_num] if solve_num >= 0 else questions)
    if is_d:
        questions_res = await solve_type_d_questions_by_openai_dependently(
            openai_client=openai_client,
            questions=question_temp,
            is_translated_to_English=is_translated_to_English,
            excel_output_path=problem_file_path,
            solve_question_prompt=solve_question_prompt,
            translate_to_english_prompt=translate_to_english_prompt,
            is_image_contained=is_image_contained,
            batch_size=batch_size,
            is_dry_run=is_dry_run,
            does_also_write_openai_answer=True,
        )
    else:
        questions_res = await solve_questions_by_openai_independently(
            openai_client=openai_client,
            questions=question_temp,
            is_translated_to_English=is_translated_to_English,
            excel_output_path=problem_file_path,
            solve_question_prompt=solve_question_prompt,
            translate_to_english_prompt=translate_to_english_prompt,
            is_image_contained=is_image_contained,
            batch_size=batch_size,
            is_dry_run=is_dry_run,
            does_also_write_openai_answer=True,
        )
    if num < EXECUTE_NUM-1:
        await asyncio.sleep(WAIT_SEC)


[{<Roles.system: 'system'>: 'Please answer this question.'}, {<Roles.user: 'user'>: '\n隣家で多数の猫を屋外飼養しており、毎年子猫が生まれて困っているとの相談があった。行政機関の動物愛護業務に携わる獣医師が行う飼い主への対応として適当でないのはどれか。 The answer options are １．近隣からの苦情を伝え、あとは自己責任で対応するよう伝える。 ２．感染症対策や猫の病気の治療を行っているかを確認する。 ３．不妊去勢手術の必要性について説明し、適正な頭数で飼養するよう伝える。 ４．飼養数を減らすため譲渡先を探すよう伝える。 ５．猫を屋内飼養するよう伝える。. Respond with only the number of your choice (e.g., 1, 2, 3, etc.).'}]
[{<Roles.system: 'system'>: 'Please answer this question.'}, {<Roles.user: 'user'>: '\nインフォームドコンセントに関する記述として最も適切なのはどれか。 The answer options are １．診断や治療の内容（効果やリスクなど）について医療者側が専門用語を駆使 してよく説明することをいう。 ２．獣医療トラブルを防止するために行うものであり、飼い主との関係構築のた めではない。 ３．獣医療行為の効果とリスク、費用に関して十分説明し、理解と合意を得るこ とである。 ４．伴侶動物においては救命を最優先し、飼い主が苦痛の軽減のための安楽死を 望んだとしても獣医師は決して動物の命を奪ってはいけない。 ５．産業動物は経済動物であるため、治療に際し農家の理解を求める必要はない。. Respond with only the number of your choice (e.g., 1, 2, 3, etc.).'}]
[{<Roles.system: 'system'>: 'Please answer this question.'}, {<Roles.user: 'user'>: '\n我が国における動物福祉（アニマルウェルフェア）の考え方に即した家畜の輸送に関する記述として正しいのはどれか

CancelledError: 

In [ ]:
Qs = questions_res if solve_num < 0 else questions_res[:solve_num]
token_sum = 0

tiktoken_model = "gpt2"

def cal_token_num(sentence):
    enc = tiktoken.get_encoding(tiktoken_model)
    tokens = enc.encode(sentence)
    return len(tokens)

for q in Qs:
    if is_translated_to_English:#not q.is_correct():
        print(q.question_sentence)
        print(q.answer_options)
        print(q.question_sentence_in_English)
        print(q.openai_answer)
        print(q.correct_answer)
        #question_token_num = cal_token_num(q.question_sentence_in_English)
        #options_token_num = cal_token_num(q.answer_options_in_English)
        #token_sum += question_token_num + options_token_num

print(token_sum)


0


In [ ]:
import datetime

dt_now = datetime.datetime.now()
print(dt_now.strftime('%Y/%m/%d %H:%M'))
print(f"model:{openai_client.model.model}")
score = calculate_accuracy(questions=questions_res)
print(f"score:{score}")

2025/10/29 09:47
model:gpt-5
score:{'accuracy': 0.9333333333333333, 'correct_num': 56, 'wrong_num': 4}


In [ ]:
"""
from src.services.image_encoder import pdf_encoder_in_base64
base64_image = pdf_encoder_in_base64("../problems/74th/academic_c/images/image1.PDF")

openai_client2 = OpenAIClient(model=Gpt4oMini)

answer =  await openai_client2.fetch_completion(
    system_prompt='you are vet.',
    user_prompt="""
#    これはなんの写真ですか？
""",
    base64_image=base64_image,
)

print(answer)
"""

',\n    base64_image=base64_image,\n)\n\nprint(answer)\n'

In [ ]:
system_prompt = OptimizedSolveQuestionPrompt2_for_essential_a_b.system_prompt
WIDTH = 70
cnt = 0
for ch in list(system_prompt.split()):
    print(ch, end=' ')
    cnt += len(ch)
    if cnt >= WIDTH:
        cnt = 0
        print()

You are tasked with answering this veterinary question. This is an examination in Japan. 
Therefore, you must refer to the laws, guidelines, and political system in Japan. Think 
deeply and thoroughly. Choose the best possible answer from the given options. Do not 
include explanations or additional text in the output. 